In [7]:
import os
import glob
import numpy as np
import cv2
from tqdm import tqdm

In [8]:
train_image_path = os.path.join("..", "data", "train", "img")
train_image_files = sorted(glob.glob(os.path.join(train_image_path, "*.tif")))

In [12]:
# Calculate the mean and std for normalization
sum_pixels = np.zeros(4)  # 4 channels
sum_squared_pixels = np.zeros(4)
max_pixel_value = np.zeros(4)
num_pixels = 0
for img_path in tqdm(train_image_files, desc="Processing images"):
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    img = img.astype(np.float32)


    if len(img.shape) == 2:  # If grayscale, expand to 4D
        img = np.expand_dims(img, axis=-1)
        img = np.repeat(img, 4, axis=-1)  # Repeat the single channel to 4

    elif img.shape[-1] != 4:
        raise ValueError(f"Unexpected number of channels: {img.shape[-1]} in {img_path}")

    sum_pixels += img.sum(axis=(0, 1))  # Sum over height & width
    sum_squared_pixels += (img ** 2).sum(axis=(0, 1))  # Sum of squares
    max_pixel_value = np.maximum(max_pixel_value, img.max(axis=(0, 1)))  # Track max per channel
    num_pixels += img.shape[0] * img.shape[1]  # Total pixel count per channel

# Compute mean and std
mean = sum_pixels / num_pixels
std = np.sqrt(sum_squared_pixels / num_pixels - mean ** 2)

print(f"Mean: {mean}")
print(f"Std: {std}")
print(f"Max Pixel Value per Channel: {max_pixel_value}")

Processing images: 100%|██████████| 32/32 [00:00<00:00, 34.97it/s]

Mean: [100.22273636 103.6747148  105.7907939  136.81767035]
Std: [25.77739219 28.44228847 33.616499   33.53157813]
Max Pixel Value per Channel: [255. 250. 252. 254.]


In [13]:
print(mean/max_pixel_value)

[0.39303034 0.41469886 0.41980474 0.53865225]


In [14]:
print(std/max_pixel_value)

[0.10108781 0.11376915 0.13339881 0.13201409]
